# Argo to CSV
This notebook is for development of a script that takes a data folder for a given float deployment and converts it to CSV for use in QuinCe. It is part of a set of scripts that will allow QuinCe to be integrated into the Argo workflow.

The final script is a standalone Python script that does not require this notebook. However, the notebook is archived as an easy starting point for future experimentation and development.

## Background
We want to be able to integrate QuinCe into the Argo data workflow, so it can be used for some data processing and QC tasks that aren't covered by existing Argo tools. It also allows the Argo data workflow to take advantage of features available in QuinCe such as processing algorithms such that they can be shared across multiple data streams and projects.

The general principle is that QuinCe can be inserted into the workflow at any point. Argo uses netCDF throughout its workflow, which QuinCe cannot read. The netCDF structure is also quite complex involving multiple files in multiple directories, so it needs to be simplified. Therefore, wherever a QuinCe step is required, the process will be:

1. Extract the information needed by QuinCe into a single CSV file.
2. Process the data in QuinCe.
3. Update the original netCDF files with new/updated values set in QuinCe.

All the netCDF data for a float deployment is stored in a directory structure. There is a sub-directory named `profiles` which contains individual netCDF files for each profile. Files in this folder contain multiple files for each profile, but QuinCe is only interested is those that start with the prefix `D` (core delayed mode data) or `BD` (BGC delayed mode data). It should be possible to update these files, and then re-generate all the other files in the folder structure based on those updated files. The above sequence therefore becomes:

1. Extract the information needed by QuinCe from the D/BD profile files into a single CSV file. _(Python script)_
2. Process the data in QuinCe.
3. Export the processed data to a new CSV file.
4. Update the D/BD netCDF files using the information in the exported CSV file. _(Python script)_
5. Regenerate the full suite of files netCDF files using the updated B/BD files. _(Argo tool to be defined)_

**History:** The Argo data system includes a history of changes made to their files as they are processed. We will need to ensure that this is updated as it writes the netCDF files. This will be explored in another notebook.

## This Notebook
This notebook will be used to develop a script to cover Step 1 of the above sequence: taking a set of netCDF files for a float deployment and generating a CSV file for use in QuinCe.

We anticipate that each float deployment will be a separate instrument in QuinCe. The instrument will expect a single CSV file containing all the required information. Although the Argo data structure involves multiple files and QuinCe is sophisticated enough to handle that, we believe that simplicity is key to reducing the chance of errors occurring in the translations.

### General approach
Because the data required is held in many files, we will create an in-memory database to store the data as it is extracted, thereby allowing the data to be 'built' in whatever order is most convenient. It can then be sorted and exported into a CSV file with a trivial query.

## The Script
The time for talk is over. We need action!

### Setup and Configuration
Imports and constants

In [ ]:
# Imports
import os
import glob
import sqlite3
from netCDF4 import Dataset
import numpy.ma as ma
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
import pandas as pd

# Constants, and items that would be considered as command line arguments in a standalone script
FLOAT_DEPLOYMENT = '6903574'

# Initialise the database
db_file = f'{FLOAT_DEPLOYMENT}.sqlite'
if os.path.exists(db_file):
    os.remove(db_file)

# Create the database in memory. We will save it to disk at the end
db = sqlite3.connect(":memory:")


## Create and Populate the Coordinates table
The basis of the database is the coordinates of the all measurements. These are fundamentally:

- Cycle Number
- Profile Number (implicit in the netCDF dimensions)
- Direction
- Pressure (equivalent to depth)

All measurements will be referenced by a combination of these values.

Other coordinate-type values (i.e. those not relating to an actual measured value) may include:

- Timestamp
- Position

These will be extracted and stored, but treated as optional.

This information is extracted from the profile files with the prefix `D`, and stored in a `coordinates` table. Each unique coordinate will get an ID, which will be referenced to place the measurements when they are read.

In [ ]:
cur = db.cursor()

# Create the coordinates table
cur.execute("DROP TABLE IF EXISTS coordinates")
cur.execute("""CREATE TABLE coordinates (
    "id" INTEGER PRIMARY KEY,
    "cycle_number" INTEGER NOT NULL,
    "profile_number" INTEGER NOT NULL,
	"direction" TEXT NOT NULL,
    "profile_sequence" INTEGER NOT NULL,
	"pressure" REAL NOT NULL,
	"timestamp" INTEGER,
	"longitude" REAL,
	"latitude" REAL,
	"source_file" TEXT NOT NULL
);""")

# For debugging coordinate extraction.
DEBUG = False
DEBUG_CYCLE_NUMBER = 38
DEBUG_PROFILE = 6

# Loop through all the 'D' files
profiles_dir = os.path.join(FLOAT_DEPLOYMENT, 'profiles')
d_files_pattern = f'D{FLOAT_DEPLOYMENT}_*.nc'
d_files = glob.glob(os.path.join(profiles_dir, d_files_pattern))

for file in tqdm(d_files):
    filename = os.path.basename(file)
    nc = Dataset(file, 'r', format='NETCDF4')

    profile_count = nc.dimensions['N_PROF'].size

    reference_time =  datetime.strptime(
        nc.variables['REFERENCE_DATE_TIME'][:].tobytes().decode('utf-8'),
        '%Y%m%d%H%M%S')
    
    # Cycle through profile numbers to get each profile in turn
    for profile in range(profile_count):
        cycle_number = nc.variables['CYCLE_NUMBER'][profile]
        profile_number = profile + 1
        direction = nc.variables['DIRECTION'][profile].tobytes().decode('utf-8')

        # Extract the timestamp
        timestamp = None
        juld = nc.variables['JULD'][profile]
        if not ma.is_masked(juld):
            timestamp = reference_time + timedelta(days=float(juld))

        # Position
        lon = nc.variables['LONGITUDE'][profile]
        if ma.is_masked(lon):
            lon = None

        lat = nc.variables['LATITUDE'][profile]
        if ma.is_masked(lat):
            lat = None

        # Now extract all the pressures and add records to the coordinates table
        profile_sequence = 0
        for pres in nc.variables['PRES'][profile,:]:
            profile_sequence += 1

            if DEBUG:
                if cycle_number == DEBUG_CYCLE_NUMBER and profile_number == DEBUG_PROFILE:
                    print(f'{cycle_number}, {profile_number}, {direction}, {profile_sequence}, {pres}, {int(timestamp.timestamp())}, {lon}, {lat}, {filename}')
            
            if not ma.is_masked(pres):
                cur.execute(f"""
                    INSERT INTO coordinates
                    (cycle_number, profile_number, direction, profile_sequence, pressure, timestamp, longitude, latitude, source_file)
                    VALUES
                    ({cycle_number}, {profile_number}, '{direction}', {profile_sequence}, {pres}, {int(timestamp.timestamp())}, {lon}, {lat}, '{filename}')""")
                db.commit()
    
    nc.close()

# Index the coordinates
dummy = cur.execute("CREATE INDEX coord_idx ON coordinates (cycle_number, profile_number, direction, profile_sequence)")


## Function to get a coordinate ID
This is used to get a coordinate ID from coordinate details

In [ ]:
# Exception thrown when a coordinate can't be found
class NoCoordinateException(Exception):
    def __init__(self, cycle, profile, direction, sequence):
        super().__init__(f'Cannot find coordinate record for {cycle} {profile} {direction} {sequence}')
        self.coord_info = f'{cycle} {profile} {direction} {sequence}'

def get_coord(cur, cycle, profile, direction, sequence):
    cur.execute(f"""SELECT id FROM coordinates WHERE cycle_number = {cycle} AND
        profile_number = {profile} AND direction = '{direction}' AND profile_sequence = {sequence}""")
    id = cur.fetchone()
    if id is None:
        raise NoCoordinateException(cycle, profile, direction, sequence)
    
    return id[0]

## Create the Data table
The data table contains all the data points from the variables we're interested in. There is one record per value, with the following fields:

- The name of the variable
- The coordinate of the value (a reference into the `coordinates` table
- The value
- The QC flag for the value

**TODO** It seems to be quite common that there are variable values measured where the `PRES` variable in the netCDF is empty. For now we ignore these and print a message. There are a lot of them. Need to decide what to do about these.

In [ ]:
cur.execute("DROP TABLE IF EXISTS data_values")
cur.execute("""CREATE TABLE data_values (
    "variable" TEXT NOT NULL,
    "coordinate" INTEGER NOT NULL,
    "value" REAL NOT NULL,
	"qc" INTEGER NOT NULL
);""")
db.commit()


## Extract data from the D files
Extract values from the required variables and add them to the data table. We look up coordinate IDs as we go.

It seems to be quite common that there are variable values measured where the `PRES` variable in the netCDF is empty. For now we ignore these. There are a lot of them. Need to decide what to do about them.

In [ ]:
D_FILE_VARIABLES = ['TEMP', 'PSAL']

skipped_count = 0

for file in tqdm(d_files):
    nc = Dataset(file, 'r', format='NETCDF4')
    profile_count = nc.dimensions['N_PROF'].size

    for var in D_FILE_VARIABLES:
        var_data = nc.variables[var]
        qc_data = nc.variables[f'{var}_QC']

        for profile in range(profile_count):
            cycle_number = nc.variables['CYCLE_NUMBER'][profile]
            profile_number = profile + 1
            direction = nc.variables['DIRECTION'][profile].tobytes().decode('utf-8')

            profile_data = var_data[profile, :]
            profile_qc = qc_data[profile, :]
            
            for i in range(len(profile_data)):
                if not ma.is_masked(profile_data[i]):
                    try:
                        coord_id = get_coord(cur, cycle_number, profile_number, direction, i + 1)
                        cur.execute("""INSERT INTO data_values (variable, coordinate, value, qc)
                            VALUES (?, ?, ?, ?)""", (var, coord_id, float(profile_data[i]), int(profile_qc[i])))
                        db.commit()
                    except NoCoordinateException as e:
                        skipped_count += 1

    nc.close()

print(f'Skipped {skipped_count} invalid coordinates')

## Add data from BD files
Add information from the biogeochemistry files. This works the same as for the D files above.

**TODO** Split this out into a function in the real script. It's a copy of the code above.

In [ ]:
bd_files_pattern = f'BD{FLOAT_DEPLOYMENT}_*.nc'
bd_files = glob.glob(os.path.join(profiles_dir, bd_files_pattern))

BD_FILE_VARIABLES = ['DOXY']

skipped_count = 0

for file in tqdm(bd_files):
    nc = Dataset(file, 'r', format='NETCDF4')
    profile_count = nc.dimensions['N_PROF'].size
    
    for var in BD_FILE_VARIABLES:
        var_data = nc.variables[var]
        qc_data = nc.variables[f'{var}_QC']

        for profile in range(profile_count):
            cycle_number = nc.variables['CYCLE_NUMBER'][profile]
            profile_number = profile + 1
            direction = nc.variables['DIRECTION'][profile].tobytes().decode('utf-8')

            profile_data = var_data[profile, :]
            profile_qc = qc_data[profile, :]
            
            for i in range(len(profile_data)):
                if not ma.is_masked(profile_data[i]):
                    try:
                        coord_id = get_coord(cur, cycle_number, profile_number, direction, i + 1)
                        cur.execute("""INSERT INTO data_values (variable, coordinate, value, qc)
                            VALUES (?, ?, ?, ?)""", (var, coord_id, float(profile_data[i]), int(profile_qc[i])))
                        db.commit()
                    except NoCoordinateException as e:
                        skipped_count += 1

    nc.close()

print(f'Skipped {skipped_count} invalid coordinates')

## Generate a DataFrame of the output
We build the CSV output in a DataFrame, because it's easier to fill in a table and then write it in one go.


In [ ]:
# Get the coordinates
output = pd.read_sql_query("""SELECT id AS COORD_ID, cycle_number AS CYCLE_NUMBER, profile_number AS NPROF,
    direction AS DIRECTION, profile_sequence AS NLEVEL, pressure AS PRES, timestamp AS TIMESTAMP,
    longitude AS LONGITUDE, latitude AS LATITUDE, source_file AS SOURCE_FILE FROM coordinates
    ORDER BY CYCLE_NUMBER, DIRECTION, NPROF, NLEVEL""", db, index_col='COORD_ID')

# Now we go through the variables
for var in tqdm(D_FILE_VARIABLES + BD_FILE_VARIABLES):
    var_df = pd.read_sql_query(f"SELECT coordinate, value as {var}, qc as {var}_QC FROM data_values WHERE variable = ?", db,
                              params=[var], index_col='coordinate')

    output = output.join(var_df, how='left')
    
    
output.to_csv(f'{FLOAT_DEPLOYMENT}.csv', index=False)

### Shutdown
Store database to disk and shut everything down

In [ ]:
cur.close()
db_disk = sqlite3.connect(db_file)
db.backup(db_disk)
db_disk.close()
db.close()